In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers peft accelerate bitsandbytes datasets trl cairosvg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 10.5 MB/s eta 0:00:00


In [ ]:
import os
os.chdir("/content/drive/MyDrive/DL Midterm")

In [ ]:
!pip install -q lxml

In [ ]:
import io
import re
import torch
import cairosvg
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

from PIL import Image
from tqdm import tqdm
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from lxml import etree

# =========================
# CONFIG
# =========================
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_PATH = "./svg-lora-adapter"
MERGED_PATH = "./svg-model-merged"

TEST_CSV = "test.csv"            # must contain: id, prompt
SUBMISSION_CSV = "submission.csv"
DEBUG_CSV = "submission_debug.csv"

PASS1_MAX_NEW_TOKENS = 1536
PASS2_MAX_NEW_TOKENS = 2048

MAX_NEW_TOKENS = 1536
BATCH_SIZE = 512

MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
RENDER_SIZE = 256

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter"
}

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256">'
    '<rect width="256" height="256" fill="white"/>'
    '</svg>'
)

# =========================
# LOAD MODEL
# =========================
tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
)

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.padding_side = "left"

# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     device_map="auto",
#     torch_dtype="auto",
# )

# model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model.eval()

# model.generation_config.temperature = None
# model.generation_config.top_p = None
# model.generation_config.top_k = None

print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("model pad:", model.config.pad_token_id)


# =========================
# SVG HELPERS
# =========================
def strip_namespace(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag

def extract_svg(text: str) -> str:
    text = text.strip()

    # Best case: full svg block exists
    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()

    # If output contains SVG: marker, keep only after that
    if "SVG:" in text:
        text = text.split("SVG:", 1)[1].strip()

    # If output contains an svg start but no closing tag, keep from <svg onward
    start = text.find("<svg")
    if start != -1:
        return text[start:].strip()

    return text

def render_svg(svg: str, size: int = RENDER_SIZE):
    try:
        png = cairosvg.svg2png(
            bytestring=svg.encode("utf-8"),
            output_width=size,
            output_height=size,
        )
        img = Image.open(io.BytesIO(png)).convert("RGB")
        return np.array(img)
    except Exception:
        return None

def validity_gate(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 0, "svg_too_long_or_empty"

    svg = svg.strip()

    if len(svg) > MAX_SVG_LENGTH:
        return 0, "svg_too_long_or_empty"

    try:
        root = ET.fromstring(svg)
    except Exception:
        return 0, "parse_failed"

    if strip_namespace(root.tag) != "svg":
        return 0, "root_not_svg"

    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)

        if tag not in ALLOWED_TAGS:
            return 0, f"disallowed_tag:{tag}"

        if tag == "path":
            path_count += 1

    if path_count > MAX_PATH_COUNT:
        return 0, "too_many_paths"

    if render_svg(svg) is None:
        return 0, "render_failed"

    return 1, "valid"

def repair_svg(svg: str) -> str:

    if not svg:
        return svg

    svg = svg.strip()

    # Remove anything before <svg
    start = svg.find("<svg")
    if start != -1:
        svg = svg[start:]

    if "SVG:" in svg:
        svg = svg.split("SVG:", 1)[-1].strip()

    # Remove junk after last </svg>
    if "</svg>" in svg:
        end = svg.rfind("</svg>")
        svg = svg[:end + len("</svg>")]

    # If missing closing tag → add it
    if "<svg" in svg and "</svg>" not in svg:
        svg += "</svg>"

    svg = re.sub(r"[A-Za-z0-9.\-]+$", "", svg)

    # Remove broken trailing fragments
    svg = re.sub(r"<[^>]*$", "", svg)

    return svg

def recover_svg_with_lxml(svg: str) -> str:

    if not svg or "<svg" not in svg:
        return svg
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg.encode("utf-8"), parser=parser)
        if root is None:
            return svg
        return etree.tostring(root, encoding="unicode")
    except Exception:
        return svg

def looks_collapsed(svg: str) -> bool:
    try:
        root = ET.fromstring(svg)
    except Exception:
        return True

    paths = [e for e in root.iter() if strip_namespace(e.tag) == "path"]
    nonempty_paths = [p for p in paths if p.attrib.get("d", "").strip()]

    if len(paths) > 0 and len(nonempty_paths) == 0:
        return True

    total_elems = sum(1 for _ in root.iter())
    if total_elems <= 1:
        return True

    return False

def accept_repaired_svg(original_svg: str, candidate_svg: str) -> bool:
    valid, _ = validity_gate(candidate_svg)
    if valid == 0:
        return False

    if looks_collapsed(candidate_svg):
        return False

    if len(candidate_svg) < 80:
        return False

    return True

def clean_svg_output(raw_text: str) -> str:
    svg = extract_svg(raw_text)

    valid, reason = validity_gate(svg)
    if valid == 1:
        return svg, "valid", svg, raw_text

    repaired = repair_svg(svg)
    valid, reason = validity_gate(repaired)
    if valid == 1:
        return repaired, "repaired", svg, raw_text

    xml = recover_svg_with_lxml(repaired)
    valid, reason = validity_gate(xml)
    if valid == 1:
        return xml, "xml", svg, raw_text

    # repaired = repair_svg(svg)
    # if accept_repaired_svg(svg, repaired):
    #     return repaired, "repaired", svg, raw_text

    # recovered = recover_svg_with_lxml(repaired)
    # if accept_repaired_svg(svg, recovered):
    #     return recovered, "xml", svg, raw_text

    return FALLBACK_SVG, "fallback", svg, raw_text

# =========================
# GENERATION
# =========================
def build_model_input(prompt: str) -> str:
    return f"Prompt: {prompt}\nSVG:\n"

# def clean_svg_output_with_reason(raw_text: str):
#     svg = extract_svg(raw_text)
#     valid, reason = validity_gate(svg)
#     if valid == 0:
#         return FALLBACK_SVG, reason, svg, raw_text
#     return svg, "valid", svg, raw_text

# def generate_svg_batch(prompts, batch_size=BATCH_SIZE):
#     results = []

#     for i in tqdm(range(0, len(prompts), batch_size)):
#         batch_prompts = prompts[i:i + batch_size]

#         inputs_text = [build_model_input(p) for p in batch_prompts]

#         inputs = tokenizer(
#             inputs_text,
#             return_tensors="pt",
#             padding=True,
#             truncation=True
#         ).to(model.device)

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens=MAX_NEW_TOKENS,
#                 do_sample=False,
#                 pad_token_id=tokenizer.pad_token_id,
#                 eos_token_id=tokenizer.eos_token_id,
#             )

#         prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
#         generated_only = [
#             outputs[j, int(prompt_lengths[j]):]
#             for j in range(outputs.shape[0])
#         ]

#         decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)
#         cleaned = [clean_svg_output(x) for x in decoded]
#         results.extend(cleaned)

#     return results

#def generate_svg_batch_debug(prompts, batch_size,max_new_tokens,do_sample=False,temperature=None,top_p = None,top_k = None):
def generate_svg_batch_debug(prompts, batch_size):
    final_svgs = []
    debug_rows = []

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]
        inputs_text = [build_model_input(p) for p in batch_prompts]

        inputs = tokenizer(
            inputs_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        # gen_kwargs = dict(
        #     max_new_tokens=max_new_tokens,
        #     do_sample=do_sample,
        #     pad_token_id=tokenizer.pad_token_id,
        #     eos_token_id=tokenizer.eos_token_id,
        # )

        # if do_sample:
        #     gen_kwargs["temperature"] = temperature
        #     gen_kwargs["top_p"] = top_p
        #     gen_kwargs["top_k"] = top_k


        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # with torch.no_grad():
        #     outputs = model.generate(**inputs, **gen_kwargs)


        prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
        generated_only = [
            outputs[j, int(prompt_lengths[j]):]
            for j in range(outputs.shape[0])
        ]

        decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)

        for prompt, raw in zip(batch_prompts, decoded):
            final_svg, reason, extracted_svg, raw_text = clean_svg_output(raw)
            final_svgs.append(final_svg)
            debug_rows.append({
                "prompt": prompt,
                "reason": reason,
                "raw_text": raw_text,
                "extracted_svg": extracted_svg,
                "final_svg": final_svg,
            })

    return final_svgs, pd.DataFrame(debug_rows)

# =========================
# FINAL SUBMISSION
# =========================
def build_submission_csv(
    test_csv_path: str = TEST_CSV,
    output_csv_path: str = SUBMISSION_CSV,
    debug_csv_path: str = DEBUG_CSV,
    batch_size: int = BATCH_SIZE,
    # retry_mode="sample",
):
    test_df = pd.read_csv(test_csv_path)
    test_df = test_df.dropna(subset=["id", "prompt"]).copy()
    test_df["prompt"] = test_df["prompt"].astype(str)

    # sample_df = test_df.iloc[:100].copy()

    # prompts = sample_df["prompt"].tolist()
    # ids = sample_df["id"].tolist()

    prompts = test_df["prompt"].tolist()
    ids = test_df["id"].tolist()

    svgs, debug_df = generate_svg_batch_debug(prompts, batch_size=BATCH_SIZE)
    print(debug_df["reason"].value_counts())

    assert len(ids) == len(svgs), f"ids={len(ids)}, svgs={len(svgs)}"

    submission_df = pd.DataFrame({
        "id": ids,
        "svg": svgs
    })

    submission_df.to_csv(output_csv_path, index=False)
    debug_df.to_csv("submission_debug.csv", index=False)
    return submission_df

    # print("Pass 1")
    # svgs_pass1, debug_df = generate_svg_batch_debug(
    #     prompts=prompts,
    #     batch_size=batch_size,
    #     max_new_tokens=PASS1_MAX_NEW_TOKENS,
    #     do_sample=False,
    # )

    # debug_df.insert(0, "id", ids)

    # fallback_mask = debug_df["stage"] == "fallback"
    # fallback_indices = debug_df.index[fallback_mask].tolist()

    # print("Pass 1 stage counts:")
    # print(debug_df["stage"].value_counts())

    # final_svgs = svgs_pass1[:]

    # if fallback_indices:
    #     retry_prompts = [prompts[i] for i in fallback_indices]

    #     print(f"Retrying {len(fallback_indices)} fallback rows with mode={retry_mode}")

    #     if retry_mode == "sample":
    #         retry_svgs, retry_debug = generate_svg_batch_debug(
    #             prompts=retry_prompts,
    #             batch_size=batch_size,
    #             max_new_tokens=PASS2_MAX_NEW_TOKENS,
    #             do_sample=True,
    #             temperature=0.2,
    #             top_p=0.9,
    #             top_k=20,
    #         )
    #     elif retry_mode == "long_greedy":
    #         retry_svgs, retry_debug = generate_svg_batch_debug(
    #             prompts=retry_prompts,
    #             batch_size=batch_size,
    #             max_new_tokens=PASS2_MAX_NEW_TOKENS,
    #             do_sample=False,
    #         )
    #     else:
    #         raise ValueError("retry_mode must be 'sample' or 'long_greedy'")

    #     recovered = 0
    #     for k, idx in enumerate(fallback_indices):
    #         retry_svg = retry_svgs[k]
    #         retry_stage = retry_debug.iloc[k]["stage"]
    #         retry_raw = retry_debug.iloc[k]["raw_text"]

    #         # only replace if retry escaped fallback
    #         if retry_stage != "fallback":
    #             final_svgs[idx] = retry_svg
    #             recovered += 1

    #             debug_df.at[idx, "stage"] = f"retry_{retry_stage}"
    #             debug_df.at[idx, "raw_text"] = retry_raw
    #             debug_df.at[idx, "final_svg"] = retry_svg
    #         else:
    #             debug_df.at[idx, "stage"] = "fallback_after_retry"

    #     print("Recovered fallback rows:", recovered)

    # submission_df = pd.DataFrame({
    #     "id": ids,
    #     "svg": final_svgs
    # })

    # submission_df.to_csv(output_csv_path, index=False)
    # debug_df.to_csv(debug_csv_path, index=False)

    # print("\nFinal stage counts:")
    # print(debug_df["stage"].value_counts())

    # final_validity = submission_df["svg"].apply(lambda x: validity_gate(x)[0]).mean()
    # print("\nFinal validity:", final_validity)
    # print("Saved submission:", output_csv_path)
    # print("Saved debug:", debug_csv_path)

    # return submission_df, debug_df

# =========================
# BUILD FINAL SUBMISSION
# =========================
submission_df = build_submission_csv(
    test_csv_path=TEST_CSV,
    output_csv_path=SUBMISSION_CSV,
    batch_size=BATCH_SIZE
)

# submission_df, debug_df = build_submission_csv(
#     test_csv_path="test.csv",
#     batch_size=512,
#     retry_mode="sample",
# )

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

pad: <|endoftext|> 151643
bos: None None
eos: <|im_end|> 151645
model pad: None


100%|██████████| 2/2 [07:42<00:00, 231.01s/it]

reason
xml         550
valid       449
repaired      1
Name: count, dtype: int64
